# Account Growth Manager — Strategic Account & Sales Analytics

**Portfolio project | Python · Pandas · NumPy · Matplotlib**

This notebook explains the Account Growth Manager role and uses a reproducible **synthetic dataset** to explore account revenue, client outsourcing, value-added services, retention, and organizational growth. All company names and figures generated below are fictional and are for demonstration only.


## 1. What does an Account Growth Manager do?

An **Account Growth Manager (AGM)** develops existing client relationships into long-term, mutually valuable partnerships. The role connects the client's business priorities with the company's services, commercial plans, and delivery capabilities. It is typically focused on growing existing accounts, rather than only acquiring brand-new customers.

Core responsibilities include:
- **Account strategy:** understand client goals, map stakeholders, and build account plans.
- **Sales and commercial management:** monitor revenue, pipeline, renewals, profitability, and expansion opportunities.
- **Client outsourcing:** identify suitable processes or capabilities a client could outsource, then coordinate a realistic service proposal with delivery teams.
- **Value-added services:** propose additional capabilities that solve a client problem, improve outcomes, or reduce effort and cost.
- **Relationship and risk management:** monitor satisfaction, resolve escalations, and protect renewals.
- **Cross-functional leadership:** coordinate sales, operations, finance, product, and delivery teams to deliver commitments.


## 2. Why is the role important?

A well-run account growth function helps an organization:
1. **Grow predictable revenue** through renewals, cross-selling, and upselling.
2. **Protect existing revenue** by spotting dissatisfaction and churn risk early.
3. **Improve profitability** by understanding service delivery costs and account margins.
4. **Increase client value** through relevant solutions rather than selling add-ons without a business case.
5. **Scale outsourcing responsibly** when the client's needs match the company's capabilities and service economics.
6. **Align growth with delivery capacity** so that expansion remains sustainable.

## 3. Vision and goals

**Vision:** become a trusted growth partner that helps clients achieve measurable business outcomes while building profitable, durable accounts for the organization.

**Practical goals:**
- Increase account revenue and qualified expansion pipeline.
- Improve gross margin and renewal rates.
- Identify outsourcing opportunities that are feasible and valuable to the client.
- Grow adoption of relevant value-added services.
- Improve client satisfaction and reduce churn risk.
- Maintain clear account plans, owners, next steps, and review cadence.

## 4. KPIs used in this example

| KPI | What it tells us |
|---|---|
| Annual revenue | Current scale of each account |
| Revenue growth potential | Estimated opportunity for future expansion; not a guaranteed forecast |
| Gross margin % | Revenue less delivery cost, divided by revenue |
| Client satisfaction | Simulated satisfaction score on a 0–100 scale |
| Renewal probability | Illustrative estimated likelihood of renewal |
| Outsourcing share | Share of account revenue associated with outsourced services |
| Value-added service revenue | Revenue from additional services that address client needs |
| Churn risk | A simple simulated risk indicator to prioritize account reviews |

> **Important:** this is a portfolio demonstration using randomly generated data, not real customer data or a validated predictive model. In a real organization, definitions should be aligned with Finance, Sales, and Delivery teams.


In [ ]:
# If needed, install the dependencies in your notebook environment:
# %pip install pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Fixed seed makes the synthetic dataset reproducible each time the notebook runs.
rng = np.random.default_rng(42)
N_ACCOUNTS = 120


In [ ]:
industries = [
    "Technology", "Healthcare", "Financial Services", "Retail",
    "Manufacturing", "Telecommunications", "Travel & Hospitality"
]
regions = ["North America", "Europe", "Asia Pacific", "India", "Middle East"]
service_models = ["Managed Services", "Staff Augmentation", "Project Delivery", "Hybrid Outsourcing"]
account_tiers = ["Strategic", "Growth", "Core"]

accounts = pd.DataFrame({
    "account_id": [f"ACCT-{i:04d}" for i in range(1, N_ACCOUNTS + 1)],
    "company_name": [f"Client Company {i:03d}" for i in range(1, N_ACCOUNTS + 1)],
    "industry": rng.choice(industries, N_ACCOUNTS),
    "region": rng.choice(regions, N_ACCOUNTS),
    "account_tier": rng.choice(account_tiers, N_ACCOUNTS, p=[0.22, 0.38, 0.40]),
    "service_model": rng.choice(service_models, N_ACCOUNTS),
    "annual_revenue_usd": np.round(rng.lognormal(mean=11.0, sigma=0.75, size=N_ACCOUNTS), -2),
    "delivery_cost_usd": np.zeros(N_ACCOUNTS),
    "growth_potential_pct": np.round(rng.uniform(0.03, 0.35, N_ACCOUNTS), 3),
    "client_satisfaction_100": np.clip(np.round(rng.normal(79, 11, N_ACCOUNTS)), 35, 100).astype(int),
    "renewal_probability_pct": np.zeros(N_ACCOUNTS),
    "outsourcing_share_pct": np.round(rng.uniform(10, 90, N_ACCOUNTS), 1),
    "value_added_service_revenue_usd": np.zeros(N_ACCOUNTS),
})

# Generate plausible delivery costs and related commercial measures.
margin_rate = rng.uniform(0.12, 0.42, N_ACCOUNTS)
accounts["delivery_cost_usd"] = np.round(accounts["annual_revenue_usd"] * (1 - margin_rate), -2)
accounts["renewal_probability_pct"] = np.clip(
    accounts["client_satisfaction_100"] * 0.72 + rng.normal(20, 8, N_ACCOUNTS), 25, 99
).round(1)
accounts["value_added_service_revenue_usd"] = np.round(
    accounts["annual_revenue_usd"] * rng.uniform(0.02, 0.18, N_ACCOUNTS), -2
)

accounts["gross_profit_usd"] = accounts["annual_revenue_usd"] - accounts["delivery_cost_usd"]
accounts["gross_margin_pct"] = (accounts["gross_profit_usd"] / accounts["annual_revenue_usd"] * 100).round(1)
accounts["estimated_growth_value_usd"] = (
    accounts["annual_revenue_usd"] * accounts["growth_potential_pct"]
).round(-2)
accounts["outsourced_revenue_usd"] = (
    accounts["annual_revenue_usd"] * accounts["outsourcing_share_pct"] / 100
).round(-2)

# Transparent illustrative rule for prioritization; not a machine-learning prediction.
accounts["churn_risk"] = np.select(
    [
        (accounts["client_satisfaction_100"] < 60) | (accounts["renewal_probability_pct"] < 55),
        (accounts["client_satisfaction_100"] < 72) | (accounts["renewal_probability_pct"] < 70),
    ],
    ["High", "Medium"],
    default="Low"
)

accounts = accounts.sort_values("annual_revenue_usd", ascending=False).reset_index(drop=True)
print(f"Synthetic dataset created: {len(accounts)} accounts")
display(accounts.head(10))


## 5. Portfolio-level performance snapshot

The next cell summarizes the generated account portfolio. Revenue and value figures are in USD and are fictional.


In [ ]:
total_revenue = accounts["annual_revenue_usd"].sum()
total_profit = accounts["gross_profit_usd"].sum()
weighted_margin = total_profit / total_revenue * 100
weighted_satisfaction = np.average(
    accounts["client_satisfaction_100"], weights=accounts["annual_revenue_usd"]
)
weighted_renewal = np.average(
    accounts["renewal_probability_pct"], weights=accounts["annual_revenue_usd"]
)

kpi_summary = pd.DataFrame({
    "Metric": [
        "Number of accounts", "Total annual revenue", "Total gross profit",
        "Portfolio gross margin", "Estimated growth value", "Value-added service revenue",
        "Revenue-weighted client satisfaction", "Revenue-weighted renewal probability",
        "High-risk accounts", "Average outsourcing share"
    ],
    "Value": [
        f"{len(accounts):,}", f"${total_revenue:,.0f}", f"${total_profit:,.0f}",
        f"{weighted_margin:.1f}%", f"${accounts['estimated_growth_value_usd'].sum():,.0f}",
        f"${accounts['value_added_service_revenue_usd'].sum():,.0f}",
        f"{weighted_satisfaction:.1f}/100", f"{weighted_renewal:.1f}%",
        f"{(accounts['churn_risk'] == 'High').sum():,}",
        f"{accounts['outsourcing_share_pct'].mean():.1f}%"
    ]
})
display(kpi_summary)


## 6. Visual analysis

These charts are intended to prompt account-review questions: which accounts contribute most revenue, where does growth potential sit, and are there clients that need retention attention?


In [ ]:
# 1) Top 10 accounts by annual revenue
top10 = accounts.nlargest(10, "annual_revenue_usd").sort_values("annual_revenue_usd")
plt.figure(figsize=(10, 6))
plt.barh(top10["company_name"], top10["annual_revenue_usd"] / 1_000_000)
plt.xlabel("Annual revenue (USD millions)")
plt.ylabel("Synthetic account")
plt.title("Top 10 Accounts by Annual Revenue")
plt.tight_layout()
plt.show()

# 2) Revenue and estimated growth value by industry
industry_summary = accounts.groupby("industry", as_index=False).agg(
    annual_revenue_usd=("annual_revenue_usd", "sum"),
    estimated_growth_value_usd=("estimated_growth_value_usd", "sum")
).sort_values("annual_revenue_usd", ascending=False)
positions = np.arange(len(industry_summary))
width = 0.38
plt.figure(figsize=(11, 6))
plt.bar(positions - width/2, industry_summary["annual_revenue_usd"] / 1_000_000, width, label="Annual revenue")
plt.bar(positions + width/2, industry_summary["estimated_growth_value_usd"] / 1_000_000, width, label="Estimated growth value")
plt.xticks(positions, industry_summary["industry"], rotation=35, ha="right")
plt.ylabel("USD millions")
plt.title("Current Revenue vs. Estimated Growth Value by Industry")
plt.legend()
plt.tight_layout()
plt.show()

# 3) Client satisfaction versus renewal probability, segmented by churn risk
plt.figure(figsize=(9, 6))
for risk in ["Low", "Medium", "High"]:
    subset = accounts[accounts["churn_risk"] == risk]
    plt.scatter(subset["client_satisfaction_100"], subset["renewal_probability_pct"], label=risk, alpha=0.75)
plt.xlabel("Client satisfaction (0–100)")
plt.ylabel("Illustrative renewal probability (%)")
plt.title("Client Health: Satisfaction vs. Renewal Probability")
plt.legend(title="Rule-based risk")
plt.tight_layout()
plt.show()

# 4) Revenue mix: outsourced services vs. value-added services
outsourcing_total = accounts["outsourced_revenue_usd"].sum()
value_added_total = accounts["value_added_service_revenue_usd"].sum()
plt.figure(figsize=(7, 5))
plt.bar(["Outsourced revenue", "Value-added service revenue"], [outsourcing_total / 1_000_000, value_added_total / 1_000_000])
plt.ylabel("USD millions")
plt.title("Illustrative Outsourcing and Value-Added Revenue")
plt.tight_layout()
plt.show()


## 7. Turn the analysis into account-growth actions

Use the synthetic results as an example of a monthly account review. In a real account, validate every recommendation with the account owner and delivery/finance data.


In [ ]:
# Prioritized account review list: high risk first, then largest estimated growth value.
priority_accounts = accounts.copy()
risk_order = {"High": 0, "Medium": 1, "Low": 2}
priority_accounts["risk_priority"] = priority_accounts["churn_risk"].map(risk_order)
priority_accounts = priority_accounts.sort_values(
    ["risk_priority", "estimated_growth_value_usd"], ascending=[True, False]
)

columns_to_review = [
    "account_id", "company_name", "industry", "account_tier",
    "annual_revenue_usd", "gross_margin_pct", "growth_potential_pct",
    "client_satisfaction_100", "renewal_probability_pct",
    "outsourcing_share_pct", "value_added_service_revenue_usd", "churn_risk"
]
display(priority_accounts[columns_to_review].head(15))


### Example action playbook

- **High churn risk:** contact the client, validate issues, agree on owners and recovery dates, and confirm renewal milestones.
- **High growth potential + healthy relationship:** build a joint account plan, identify client-approved use cases, and qualify expansion opportunities.
- **Low gross margin:** review service scope, staffing mix, delivery effort, pricing assumptions, and contractual commitments before proposing changes.
- **Low outsourcing share:** explore outsourcing only if it solves a client problem and delivery can meet quality, security, and compliance needs.
- **Low value-added service adoption:** run a discovery conversation and propose only services with a clear client outcome and measurable success criteria.

## 8. Export the dataset

Run the next cell to save the synthetic account dataset and summary table beside this notebook.


In [ ]:
accounts.to_csv("account_growth_synthetic_dataset.csv", index=False)
kpi_summary.to_csv("account_growth_kpi_summary.csv", index=False)
print("Saved: account_growth_synthetic_dataset.csv")
print("Saved: account_growth_kpi_summary.csv")


## 9. Final takeaway

Account growth is more than increasing sales. It combines **client trust, commercial discipline, delivery performance, outsourcing strategy, value-added solutions, and measurable business outcomes**. Analytics helps a manager focus attention, but account conversations and validated operational data are essential before making real business decisions.

**Portfolio note:** this notebook intentionally uses fictional records. Do not present its generated metrics as actual employer or client results.
